# Badge 3: Data Application Builders Workshop: Lesson 11: Using API Data With Variables

SEARCH_ON and the final DORA records. The SniS (Streamlit not in Snowflake) code changes run outside Snowflake.

**How to read this notebook**
- Each `## Page:` heading is one page of the Rise lesson, in course order.
- `> Course:` cells quote what the learner is told to do (text or screenshot steps).
- Code cells run that task. Course code is verbatim except role/warehouse names, swapped for `BADGE3_DABW_ADMIN` (ACCOUNTADMIN), `BADGE3_DABW_ROLE` (SYSADMIN) and `BADGE3_DABW_WH` (COMPUTE_WH).
- **SQL equivalent** replaces a UI-only step; **From screenshot** marks code the course shows only as an image; **Derived solution** marks challenge answers the course does not give.
- DORA checks are not run here; see `dora/dora_tests.ipynb`.

Prerequisite: run `dora/account_setup.ipynb` once.

In [ ]:
# Isolation mapping (set once per notebook; SQL cells use these via {{...}})
ADMIN_ROLE    = "BADGE3_DABW_ADMIN"   # stands in for the course's ACCOUNTADMIN
SYSADMIN_ROLE = "BADGE3_DABW_ROLE"    # stands in for the course's SYSADMIN
WAREHOUSE     = "BADGE3_DABW_WH"      # stands in for the course's COMPUTE_WH
import ast, io
from snowflake.snowpark.context import get_active_session

CUSTOM = 'SMOOTHIES.PUBLIC."Custom Smoothie Order Form"'
PENDING = 'SMOOTHIES.PUBLIC."Pending Smoothie Orders"'


def pdf(result):
    """A SQL cell's result as pandas (runtime 2.6+ returns a Snowpark DataFrame)."""
    return result.to_pandas() if hasattr(result, "to_pandas") else result


def write_app(app, app_code):
    """SQL equivalent of editing the app in Snowsight's Streamlit editor: write streamlit_app.py into the app's live version."""
    ast.parse(app_code)  # the code must at least be valid Python
    path = f"snow://streamlit/{app}/versions/live/streamlit_app.py"
    get_active_session().file.put_stream(io.BytesIO(app_code.encode()), path, auto_compress=False, overwrite=True)
    print(f"wrote {len(app_code.splitlines())} lines to {path}")

In [ ]:
%%sql -r session_context
use role {{SYSADMIN_ROLE}};
use secondary roles none;
use warehouse {{WAREHOUSE}};
select current_role(), current_warehouse(), current_database(), current_schema();

## Page: 🧪 Take the Keys

Quiz only.

## Page: 📩 Bringing Data into the App via API

> **Course:** Add the `requests` library to requirements.txt and streamlit_app.py of your SniS project and call the SmoothieFroot API (screenshots 2026011615430-440). Outside Snowflake; nothing to run.

## Page: 📊 Using SmoothieFroot Data More Effectively

> **Course:** Rearrange the SniS app's code to show nutrition data for the fruits chosen. Outside Snowflake; nothing to run.

## Page: 👨‍🍳 Using the Ingredients Chosen in the API Call

> **Course:** Use the fruit_chosen variable in the API call. Outside Snowflake; nothing to run.

## Page: 📋 Requirements Negotiation

Reading only.

## Page: ✅ Updating the Multi-Select Part 1

> **Course:** 🎯 Let's Add the SEARCH_ON Field to Our FRUIT-OPTIONS Table: add the column SEARCH_ON; copy FRUIT_NAME into it with an UPDATE; then fix the rows that need a different value (at least Apples, Blueberries, Jack Fruit, Raspberries and Strawberries; Ximenia is in SmoothieFroot, just written differently). Screenshots 2026011616025.

In [ ]:
%%sql -r add_search_on
-- Derived solution (screenshot 2026011616025.jpg shows the shape)
use role {{SYSADMIN_ROLE}};
alter table smoothies.public.fruit_options add column SEARCH_ON varchar(25);

In [ ]:
%%sql -r seed_search_on
--copy name to seed the columns - then we can just edit the problem rows
update smoothies.public.fruit_options
set search_on = fruit_name;

In [ ]:
%%sql -r fix_search_on
-- The values below are the names https://my.smoothiefroot.com/api/fruit/all returns (checked 2026-11-19)
update smoothies.public.fruit_options set search_on = 'Apple' where fruit_name = 'Apples';
update smoothies.public.fruit_options set search_on = 'Blueberry' where fruit_name = 'Blueberries';
update smoothies.public.fruit_options set search_on = 'Raspberry' where fruit_name = 'Raspberries';
update smoothies.public.fruit_options set search_on = 'Strawberry' where fruit_name = 'Strawberries';
update smoothies.public.fruit_options set search_on = 'Ximenia (Hog Plum)' where fruit_name = 'Ximenia';

In [ ]:
%%sql -r check_search_on
select * from smoothies.public.fruit_options order by fruit_id;

## Page: ✅ Updating the Multi-Select Part 2

> **Course:** Add the SEARCH_ON column to the dataframe that feeds the multiselect, bring in pandas as pd, make `pd_df`, and use the loc/iloc statement (screenshots 2026011616040-050). Outside Snowflake (SniS); nothing to run.

## Page: ✨ Putting on the Finishing Touches

> **Course:** Preparing Records for Your Final DORA Check: create three orders using your app, with the fruits in the order described. Kevin: Apples, Lime, Ximenia (NOT filled). Divya: Dragon Fruit, Guava, Figs, Jackfruit, Blueberries (FILLED). Xi: Vanilla Fruit, Nectarine (FILLED). If you mess this up you can truncate the table and start over.

In [ ]:
%%sql -r truncate_orders
-- the course allows a truncate to start over
truncate table smoothies.public.orders;

In [ ]:
%%sql -r order_kevin
-- the statement the ordering app builds on Submit Order (each fruit followed by a space)
insert into smoothies.public.orders(ingredients, name_on_order) values ('Apples Lime Ximenia ','Kevin');

In [ ]:
%%sql -r order_divya
insert into smoothies.public.orders(ingredients, name_on_order) values ('Dragon Fruit Guava Figs Jackfruit Blueberries ','Divya');

In [ ]:
%%sql -r order_xi
insert into smoothies.public.orders(ingredients, name_on_order) values ('Vanilla Fruit Nectarine ','Xi');

In [ ]:
%%sql -r mark_filled
-- SQL equivalent of ticking ORDER_FILLED for Divya and Xi in the Pending Smoothie Orders app and clicking Submit
update smoothies.public.orders set order_filled = true where name_on_order in ('Divya','Xi');

In [ ]:
%%sql -r final_orders
select * from smoothies.public.orders order by order_uid;

## Page: 🤖 DORA DABW008

### 🤖 DORA check DABW008 goes here
The course has the learner run **DABW008** (the Kevin, Divya and Xi orders) at this point.
It is in `dora/dora_tests.ipynb`, section **DABW008**, as a local check plus the exact DORA code.

## Page: 🎉 Badge Time!

> **Course:** Check in with DORA in the YSA app: DABW001 to DABW008 should be PASSING and VALID. Outside Snowflake; nothing to run.